<a href="https://colab.research.google.com/github/juanpajedrez/pytorch_learning/blob/main/08_pytorch_paper_replicating_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08. PyTorch Paper Replicating Exercises

Welcome to the 08. PyTorch Paper Replicating exercises.

Your objective is to write code to satisify each of the exercises below.

Some starter code has been provided to make sure you have all the resources you need.

> **Note:** There may be more than one solution to each of the exercises.

## Resources

1. These exercises/solutions are based on [section 08. PyTorch Paper Replicating](https://www.learnpytorch.io/08_pytorch_paper_replicating/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.
2. See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/tjpW_BY8y3g) (but try the exercises yourself first!).
3. See [all solutions on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/extras/solutions).

> **Note:** The first section of this notebook is dedicated to getting various helper functions and datasets used for the exercises. The exercises start at the heading "Exercise 1: ...".

### Get various imports and helper functions

The code in the following cells prepares imports and data for the exercises below. They are taken from [08. PyTorch Paper Replicating](https://www.learnpytorch.io/08_pytorch_paper_replicating/).

In [ ]:
# For this notebook to run with updated APIs, we need torch 1.12+ and torchvision 0.13+
try:
    import torch
    import torchvision
    assert int(torch.__version__.split(".")[1]) >= 12, "torch version should be 1.12+"
    assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")
except:
    print(f"[INFO] torch/torchvision versions not as required, installing nightly versions.")
    !pip3 install -U --pre torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/nightly/cu113
    import torch
    import torchvision
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")


In [ ]:
# Continue with regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
    from going_modular.going_modular import data_setup, engine
    from helper_functions import download_data, set_seeds, plot_loss_curves
except:
    # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular or helper_functions scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !mv pytorch-deep-learning/helper_functions.py . # get the helper_functions.py script
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine
    from helper_functions import download_data, set_seeds, plot_loss_curves

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
print(torch.__version__)

### Get data

Want to download the data we've been using in PyTorch Paper Replicating: https://www.learnpytorch.io/08_pytorch_paper_replicating/#1-get-data

In [ ]:
# Download pizza, steak, sushi images from GitHub
image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

In [ ]:
# Setup directory paths to train and test images
train_dir = image_path / "train"
test_dir = image_path / "test"

### Preprocess data

Turn images into tensors using same code as PyTorch Paper Replicating section 2.1 and 2.2: https://www.learnpytorch.io/08_pytorch_paper_replicating/#21-prepare-transforms-for-images

In [ ]:
# Create image size (from Table 3 in the ViT paper)
IMG_SIZE = 224

# Create transform pipeline manually
manual_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])
print(f"Manually created transforms: {manual_transforms}")

In [ ]:
# Set the batch size
BATCH_SIZE = 32 # this is lower than the ViT paper but it's because we're starting small

# Create data loaders
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=manual_transforms, # use manually created transforms
    batch_size=BATCH_SIZE
)

train_dataloader, test_dataloader, class_names

In [ ]:
# Get a batch of images
image_batch, label_batch = next(iter(train_dataloader))

# Get a single image from the batch
image, label = image_batch[0], label_batch[0]

# View the batch shapes
image.shape, label

In [ ]:
# Plot image with matplotlib
plt.imshow(image.permute(1, 2, 0)) # rearrange image dimensions to suit matplotlib [color_channels, height, width] -> [height, width, color_channels]
plt.title(class_names[label])
plt.axis(False);

## 1. Replicate the ViT architecture we created with in-built [PyTorch transformer layers](https://pytorch.org/docs/stable/nn.html#transformer-layers).

* You'll want to look into replacing our `TransformerEncoderBlock()` class with [`torch.nn.TransformerEncoderLayer()`](https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html#torch.nn.TransformerEncoderLayer) (these contain the same layers as our custom blocks).
* You can stack `torch.nn.TransformerEncoderLayer()`'s on top of each other with [`torch.nn.TransformerEncoder()`](https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html#torch.nn.TransformerEncoder).

Need:
1. `PatchEmbedding` (turn images into embedded patches).
2. Transformer Encoder layer (this is comprised of alternating MSA and MLP blocks).
3. Stack multiple transformer encoder layers on top of each other.
4. MLP head.
5. Put it all together to create ViT

### 1. Make PatchEmbedding Layer

In [ ]:
#1. Create a class which subclasses nn.Module
class PatchEmbedding(nn.Module):
  """Turns a 2D input image into a 1D sequence learnable embedding vector.

    Args:
        in_channels (int): Number of color channels for the input images. Defaults to 3.
        patch_size (int): Size of patches to convert input image into. Defaults to 16.
        embedding_dim (int): Size of embedding to turn image into. Defaults to 768.
  """

  # 2. Initialize the class with appropiate variables
  def __init__(self,
               in_channels:int = 3,
               patch_size:int = 16,
               embedding_dim:int= 768):
      super().__init__()

      # 3. Create a layer to turn an image into patches
      self.patcher = nn.Conv2d(in_channels = in_channels,
                               out_channels = embedding_dim,
                               kernel_size=patch_size,
                               stride = patch_size,
                               padding = 0)

      # 4. Create a layer to flatten the patch feature maps into a single dimension
      self.flatten = nn.Flatten(start_dim=2, # Only flatten the feature map dimensions into a single vector
                                end_dim=3)

      # To set it up so it can divide
      self.patch_size = patch_size

  # 5. Define the forward method
  def forward(self, x):
    # Create assertion to check that inputs are the correct shape
    image_resolution = x.shape[-1]
    assert image_resolution % self.patch_size == 0, f"Input image size must be divisible by patch size, image shape: {image_resolution}, patch size: {self.patch_size}"

    # Perform the forward pass
    x_patched = self.patcher(x)
    x_flattened = self.flatten(x_patched)

    # 6. Make sure the output shape has the right order
    return x_flattened.permute(0, 2, 1) # adjust so the embedding is on the final dimension [batch_size, P^2•C, N] -> [batch_size, N, P^2•C]

### 2. Transformer Encoder Layer

Can build a transformer Encoder Layer with:
https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html

In [ ]:
rand_image_tensor = torch.rand(32, 3, 224, 224) # (batch_size, colour_channels, height, width)
rand_image_tensor.shape

In [ ]:
patch_embedding = PatchEmbedding()
patch_embedding_output = patch_embedding(rand_image_tensor)
print(f"Input shape: {rand_image_tensor.shape}")
print(f"Output shape: {patch_embedding_output.shape}")

In [ ]:
transformer_encoder_layer = nn.TransformerEncoderLayer(
    d_model = 768, # embedding dimension
    nhead = 12,
    dim_feedforward = 3072,
    dropout = 0.1,
    activation = 'gelu',
    batch_first=True,
    norm_first=True
)
transformer_encoder_layer

In [ ]:
from torchinfo import summary
summary(model = transformer_encoder_layer,
        input_size=patch_embedding_output.shape, # (batch_size, number_of_patches + 1 positional embedding, embedding_dim)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"]
        )

### 3. Stack Transformer Encoder Layers on Top of each other to make the full transformer Encoder

According to Table 1 of the ViT paper, the ViT-Base model uses a stack of 12 Transformer Encoder Layers.

We can stack Transformer Encoder Layers on top of each other using: https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html

In [ ]:
transformer_encoder = nn.TransformerEncoder(
    encoder_layer = transformer_encoder_layer,
    num_layers = 12)

In [ ]:
from torchinfo import summary
# summary(model = transformer_encoder,
#         input_size=(1, 196, 768), # (batch_size, number_of_patches + 1 positional embedding, embedding_dim)
#         col_names = ["input_size", "output_size", "num_params", "trainable"],
#         col_width = 20,
#         row_settings = ["var_names"]
#         )

### 5. Put it all together and create ViT

We're skipping step 4, so it can be incorporated into our overall ViT architecture.

In [ ]:
class ViT(nn.Module):
  def __init__(self,
               img_size :int = 224, # from Table 3
               num_channels:int = 3,
               patch_size: int = 16,
               embedding_dim: int = 768, # from Table 1
               dropout:float = 0.1,
               mlp_size:int = 3072, # from table 1
               num_transformer_layers:int= 12, # from table 1
               num_heads:int =12, # from Table 1 (number of multi head self attention heads)
               num_classes:int = 1000): # generic number of classes (this can be adjusted)

    super().__init__()

    # Assert image size is divisble by patch size
    assert img_size % patch_size == 0, f"Image size must be divisible by patch size, image size: {img_size}, patch"

    # 1. Create patch embedding
    self.patch_embedding = PatchEmbedding(in_channels=num_channels,
                                          patch_size=patch_size,
                                          embedding_dim=embedding_dim)

    # 2. Create class token
    self.class_token = nn.Parameter(torch.randn(1, 1, embedding_dim), requires_grad=True)

    # 3. Create positional embedding
    num_patches = (img_size * img_size) // (patch_size ** 2) # N = HW/P^2
    self.positional_embedding = nn.Parameter(torch.randn(1, 1 + num_patches, embedding_dim), requires_grad=True)

    # 4. Create patch + position embedding dropout
    self.embedding_dropout = nn.Dropout(p = dropout)

    # 5. Create Transformer Encoder Layer
    # self.transformer_encoder_layer = nn.TransformerEncoderLayer(d_model = embedding_dim,
    #                                                             nhead=num_heads,
    #                                                             dim_feedforward=mlp_size,
    #                                                             dropout=dropout,
    #                                                             activation='gelu',
    #                                                             batch_first=True)

    # 6. Create stack of Transformer Encoder Layers
    self.transformer_encoder = nn.TransformerEncoder(encoder_layer=nn.TransformerEncoderLayer(
                                                                    d_model = embedding_dim,
                                                                    nhead=num_heads,
                                                                    dim_feedforward=mlp_size,
                                                                    dropout=dropout,
                                                                    activation='gelu',
                                                                    batch_first=True),
                                                     num_layers=num_transformer_layers)

    # 7. Create MLP head
    self.mlp_head = nn.Sequential(
        nn.LayerNorm(normalized_shape=embedding_dim),
        nn.Linear(in_features=embedding_dim,
                  out_features = num_classes),
    )

  def forward(self, x):
    # Get some dimensions from x
    batch_size = x.shape[0]

    # Create the patch embedding
    x = self.patch_embedding(x)

    # Expand the class token across each batches -> class token per image.
    class_token = self.class_token.expand(batch_size, -1, -1)

    # Prepend the class token to the patch embedding
    x = torch.cat((class_token, x), dim=1)

    # Add the positional embedding to patch embedding with class token
    x = self.positional_embedding + x

    # Dropout on patch + positional embedding
    x = self.embedding_dropout(x)

    #Pass embedding through Transformer Encoder stack
    x = self.transformer_encoder(x)

    # Pass the 0th endex o x through MLP head
    x = self.mlp_head(x[:, 0]) # We want all the batches

    return x


In [ ]:
demo_img = torch.rand(1, 3, 224, 224).to(device)
vit = ViT(num_channels=len(class_names)).to(device)
vit(demo_img)

In [ ]:
summary(model=vit,
        input_size=(1, 3, 224, 224),
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"])

In [ ]:
embedding_dim = 768
class_token = nn.Parameter(torch.randn(1, 1, embedding_dim), requires_grad=True)
class_token.requires_grad

In [ ]:
batch_size = 32
print(class_token.shape)
class_token.expand(batch_size, -1, -1).shape # '-1' means to infer the dimension, whatever they were before

In [ ]:
patch_size = 16
img_size = 224
num_patches = (img_size * img_size) // (patch_size ** 2)
pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, embedding_dim), requires_grad=True)
pos_embedding.shape

## 2. Turn the custom ViT architecture we created into a Python script, for example, `vit.py`.

* You should be able to import an entire ViT model using something like`from vit import ViT`.
* We covered the art of turning code cells into Python scrips in [05. PyTorch Going Modular](https://www.learnpytorch.io/05_pytorch_going_modular/).


Let's copy all of our ViT model dependencies to a single cell and write it to file using the magic

```python
%writefile FILENAME
```

In [ ]:
%%writefile vit.py
import torch
from torch import nn

#1. Create a class which subclasses nn.Module
class PatchEmbedding(nn.Module):
  """Turns a 2D input image into a 1D sequence learnable embedding vector.

    Args:
        in_channels (int): Number of color channels for the input images. Defaults to 3.
        patch_size (int): Size of patches to convert input image into. Defaults to 16.
        embedding_dim (int): Size of embedding to turn image into. Defaults to 768.
  """

  # 2. Initialize the class with appropiate variables
  def __init__(self,
               in_channels:int = 3,
               patch_size:int = 16,
               embedding_dim:int= 768):
      super().__init__()

      # 3. Create a layer to turn an image into patches
      self.patcher = nn.Conv2d(in_channels = in_channels,
                               out_channels = embedding_dim,
                               kernel_size=patch_size,
                               stride = patch_size,
                               padding = 0)

      # 4. Create a layer to flatten the patch feature maps into a single dimension
      self.flatten = nn.Flatten(start_dim=2, # Only flatten the feature map dimensions into a single vector
                                end_dim=3)

      # To set it up so it can divide
      self.patch_size = patch_size

  # 5. Define the forward method
  def forward(self, x):
    # Create assertion to check that inputs are the correct shape
    image_resolution = x.shape[-1]
    assert image_resolution % self.patch_size == 0, f"Input image size must be divisible by patch size, image shape: {image_resolution}, patch size: {self.patch_size}"

    # Perform the forward pass
    x_patched = self.patcher(x)
    x_flattened = self.flatten(x_patched)

    # 6. Make sure the output shape has the right order
    return x_flattened.permute(0, 2, 1) # adjust so the embedding is on the final dimension [batch_size, P^2•C, N] -> [batch_size, N, P^2•C]


class ViT(nn.Module):
  def __init__(self,
               img_size :int = 224, # from Table 3
               num_channels:int = 3,
               patch_size: int = 16,
               embedding_dim: int = 768, # from Table 1
               dropout:float = 0.1,
               mlp_size:int = 3072, # from table 1
               num_transformer_layers:int= 12, # from table 1
               num_heads:int =12, # from Table 1 (number of multi head self attention heads)
               num_classes:int = 1000): # generic number of classes (this can be adjusted)

    super().__init__()

    # Assert image size is divisble by patch size
    assert img_size % patch_size == 0, f"Image size must be divisible by patch size, image size: {img_size}, patch"

    # 1. Create patch embedding
    self.patch_embedding = PatchEmbedding(in_channels=num_channels,
                                          patch_size=patch_size,
                                          embedding_dim=embedding_dim)

    # 2. Create class token
    self.class_token = nn.Parameter(torch.randn(1, 1, embedding_dim), requires_grad=True)

    # 3. Create positional embedding
    num_patches = (img_size * img_size) // (patch_size ** 2) # N = HW/P^2
    self.positional_embedding = nn.Parameter(torch.randn(1, 1 + num_patches, embedding_dim), requires_grad=True)

    # 4. Create patch + position embedding dropout
    self.embedding_dropout = nn.Dropout(p = dropout)

    # 5. Create Transformer Encoder Layer
    # self.transformer_encoder_layer = nn.TransformerEncoderLayer(d_model = embedding_dim,
    #                                                             nhead=num_heads,
    #                                                             dim_feedforward=mlp_size,
    #                                                             dropout=dropout,
    #                                                             activation='gelu',
    #                                                             batch_first=True)

    # 6. Create stack of Transformer Encoder Layers
    self.transformer_encoder = nn.TransformerEncoder(encoder_layer=nn.TransformerEncoderLayer(
                                                                    d_model = embedding_dim,
                                                                    nhead=num_heads,
                                                                    dim_feedforward=mlp_size,
                                                                    dropout=dropout,
                                                                    activation='gelu',
                                                                    batch_first=True),
                                                     num_layers=num_transformer_layers)

    # 7. Create MLP head
    self.mlp_head = nn.Sequential(
        nn.LayerNorm(normalized_shape=embedding_dim),
        nn.Linear(in_features=embedding_dim,
                  out_features = num_classes),
    )

  def forward(self, x):
    # Get some dimensions from x
    batch_size = x.shape[0]

    # Create the patch embedding
    x = self.patch_embedding(x)

    # Expand the class token across each batches -> class token per image.
    class_token = self.class_token.expand(batch_size, -1, -1)

    # Prepend the class token to the patch embedding
    x = torch.cat((class_token, x), dim=1)

    # Add the positional embedding to patch embedding with class token
    x = self.positional_embedding + x

    # Dropout on patch + positional embedding
    x = self.embedding_dropout(x)

    #Pass embedding through Transformer Encoder stack
    x = self.transformer_encoder(x)

    # Pass the 0th endex o x through MLP head
    x = self.mlp_head(x[:, 0]) # We want all the batches

    return x

In [ ]:
from vit import ViT

imported_vit = ViT()
summary(model = imported_vit,
        input = (1, 3, 224, 224))

## 3. Train a pretrained ViT feature extractor model (like the one we made in [08. PyTorch Paper Replicating section 10](https://www.learnpytorch.io/08_pytorch_paper_replicating/#10-bring-in-pretrained-vit-from-torchvisionmodels-on-same-dataset)) on 20% of the pizza, steak and sushi data like the dataset we used in [07. PyTorch Experiment Tracking section 7.3](https://www.learnpytorch.io/07_pytorch_experiment_tracking/#73-download-different-datasets)
* See how it performs compared to the EffNetB2 model we compared it to in [08. PyTorch Paper Replicating section 10.6](https://www.learnpytorch.io/08_pytorch_paper_replicating/#106-save-feature-extractor-vit-model-and-check-file-size).

In [ ]:
set_seeds()

In [ ]:
# Create ViT feature extractor model
import torchvision

# Download pretrained ViT weights and model
vit_weights = torchvision.models.ViT_B_16_Weights.DEFAULT # 'DEFAULTS' means best available
pretrained_vit = torchvision.models.vit_b_16(weights=vit_weights).to(device)

# Freeze all layers in pretrained ViT model
for param in pretrained_vit.parameters():
    param.requires_grad = False

# Update the pretrained ViT head
embedding_dim = 768
pretrained_vit.heads = nn.Sequential(
    nn.LayerNorm(normalized_shape=embedding_dim),
    nn.Linear(
      in_features=embedding_dim,
      out_features=len(class_names))
)

In [ ]:
summary(model = pretrained_vit,
        input_size= (1, 3, 224, 224),
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"])

In [ ]:
test_dir

In [ ]:
# Get 20% of data
data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

# Setup train and test directories
train_dir_20_percent = data_20_percent_path / "train"
#test_dir_20_percent = data_20_percent_path / "test" don't nered 20% test data as the model in 07. PyTorch Experiment Tracking section 7.3 tests on the 10% of the dataset

# Preprocess the data
vit_transforms = vit_weights.transforms()
train_dataloader_20_percent, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir_20_percent,
                                                                                          test_dir = test_dir, # use 10% data for testing
                                                                                          transform = vit_transforms,
                                                                                          batch_size = 1024)

In [ ]:
len(train_dataloader), len(train_dataloader_20_percent), len(test_dataloader)

In [ ]:
# Train a pretrained ViT feature extractor
from going_modular.going_modular import engine

optimizer = torch.optim.Adam(pretrained_vit.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

set_seeds()
epochs = 10
pretrained_vit_results = engine.train(model = pretrained_vit,
                                      train_dataloader = train_dataloader_20_percent,
                                      test_dataloader = test_dataloader,
                                      optimizer = optimizer,
                                      loss_fn = loss_fn,
                                      epochs = epochs,
                                      device = device)

In [ ]:
# Examine results
from helper_functions import plot_loss_curves
plot_loss_curves(pretrained_vit_results)

## 4. Try repeating the steps from excercise 3 but this time use the "`ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1`" pretrained weights from [`torchvision.models.vit_b_16()`](https://pytorch.org/vision/stable/models/generated/torchvision.models.vit_b_16.html#torchvision.models.vit_b_16).
* Note: ViT pretrained with SWAG weights has a minimum input image size of (384, 384), though this is accessible in the weights `.transforms()` method.

* Note: ViT pretrained with SWAG Weights has a minimum input image size of (384, 384). This is accessible in weights `.transforms()` method.

In [ ]:
# Create ViT feature extractor model
import torchvision

# Download pretrained ViT weights and model
vit_weights_swag = torchvision.models.ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1 # get swag weights
pretrained_vit_swag = torchvision.models.vit_b_16(weights=vit_weights_swag).to(device)

# Freeze all layers in pretrained ViT model
for param in pretrained_vit_swag.parameters():
    param.requires_grad = False

# Update the pretrained ViT head
embedding_dim = 768
pretrained_vit_swag.heads = nn.Sequential(
    nn.LayerNorm(normalized_shape=embedding_dim),
    nn.Linear(
      in_features=embedding_dim,
      out_features=len(class_names))
)

#P Print a summary
summary(model = pretrained_vit_swag,
        input_size= (1, 3, 384, 384),
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"])

ViT pretrained SWAG weights has a minimum inpu image size of (384, 384), this is accessible through the `.transforms()` method, following the link: https://docs.pytorch.org/vision/main/models/generated/torchvision.models.vit_b_16.html#torchvision.models.vit_b_16

In [ ]:
# Checlkout transforms for pretrained ViT with Swag Weights
vit_transforms_swag = vit_weights_swag.transforms()
vit_transforms_swag

In [ ]:
# Get 20% of data
data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

# Setup train and test directories
train_dir_20_percent = data_20_percent_path / "train"
#test_dir_20_percent = data_20_percent_path / "test" don't nered 20% test data as the model in 07. PyTorch Experiment Tracking section 7.3 tests on the 10% of the dataset

# Preprocess the data
train_dataloader_20_percent, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir_20_percent,
                                                                                          test_dir = test_dir, # use 10% data for testing
                                                                                          transform = vit_transforms_swag,
                                                                                          batch_size = 1024)

In [ ]:
# Train a pretrained ViT feature extractor with SWAG Weights
from going_modular.going_modular import engine

optimizer = torch.optim.Adam(pretrained_vit_swag.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

set_seeds()
epochs = 10
pretrained_vit_swag_results = engine.train(model = pretrained_vit_swag,
                                      train_dataloader = train_dataloader_20_percent,
                                      test_dataloader = test_dataloader,
                                      optimizer = optimizer,
                                      loss_fn = loss_fn,
                                      epochs = epochs,
                                      device = device)

In [ ]:
# Examine results
from helper_functions import plot_loss_curves
plot_loss_curves(pretrained_vit_swag_results)

# 5. Our custom ViT model architecture closely mimics that of the ViT paper, however, our training recipe misses a few things.
* Research some of the following topics from Table 3 in the ViT paper that we miss and write a sentence about each and how it might help with training:
    * **ImageNet-21k pretraining**
    * **Learning rate warmup**
    * **Learning rate decay**
    * **Gradient clipping**

In [ ]:
# TODO: your explanations of the above terms